In [113]:
import os

In [114]:
%pwd

'c:\\Users\\Astha\\Chicken-Disease-Classification-Project'

In [115]:
os.chdir("c:\\Users\\Astha\\Chicken-Disease-Classification-Project")

In [116]:
%pwd

'c:\\Users\\Astha\\Chicken-Disease-Classification-Project'

In [117]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list


@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path



In [118]:
print(TrainingConfig)
print(TrainingConfig.__annotations__)

<class '__main__.TrainingConfig'>
{'root_dir': <class 'pathlib.Path'>, 'trained_model_path': <class 'pathlib.Path'>, 'updated_base_model_path': <class 'pathlib.Path'>, 'training_data': <class 'pathlib.Path'>, 'params_epochs': <class 'int'>, 'params_batch_size': <class 'int'>, 'params_is_augmentation': <class 'bool'>, 'params_image_size': <class 'list'>}


In [119]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [120]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH): 
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_prepare_callbacks_config(self) -> PrepareCallbacksConfig:
        config = self.config.prepare_callbacks
        model_ckpt_dir=os.path.dirname(config.checkpoint_model_filepath)
        

        create_directories([
            Path(model_ckpt_dir),
            Path(config.tensorboard_root_log_dir)
        ]) 

        prepare_callbacks_config = PrepareCallbacksConfig(
            root_dir=Path(config.root_dir),
            tensorboard_root_log_dir=Path(config.tensorboard_root_log_dir),
            checkpoint_model_filepath=Path(config.checkpoint_model_filepath)
        )

        return prepare_callbacks_config
    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "Chicken-fecal-images")
        create_directories([
            Path(training.root_dir)
            ])
        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )
        return training_config
   

In [121]:
import time

In [122]:
class PrepareCallbacks:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config

    @property
    def _create_tb_callbacks(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_running_log_dir= os.path.join(
            self.config.tensorboard_root_log_dir,
            f"tb_logs_at_{timestamp}",
        )
        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)
    
    @property
    def _create_ckpt_callbacks(self):
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=self.config.checkpoint_model_filepath,
            save_best_only=True
        )
    
    def get_tb_ckpt_callbacks(self):
        return[
            self._create_tb_callbacks,
            self._create_ckpt_callbacks
        ]


In [123]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [125]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
        self.config.updated_base_model_path,
        compile=False
    )

        self.model.compile(
            optimizer=tf.keras.optimizers.SGD(
                learning_rate=0.01
         ),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )
    
    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                rotation_range=20,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        
        self.train_generator = train_datagenerator.flow_from_directory(
                directory=self.config.training_data,
                subset="training",
                shuffle=True,
                **dataflow_kwargs

            )
    @staticmethod
    def save_model(path:Path,model:tf.keras.Model):
        model.save(path)

    def train(self,callback_list: list):
        self.steps_per_epoch= self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps= self.valid_generator.samples // self.valid_generator.batch_size


        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator,
            callbacks=callback_list
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )




In [127]:
pip install tensorboard

   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.5 MB 1.9 MB/s eta 0:00:03
   ---------------------------------------- 0.1/5.5 MB 435.7 kB/s eta 0:00:13
    --------------------------------------- 0.1/5.5 MB 1.2 MB/s eta 0:00:05
   - -------------------------------------- 0.3/5.5 MB 1.4 MB/s eta 0:00:04
   -- ------------------------------------- 0.3/5.5 MB 1.5 MB/s eta 0:00:04
   --- ------------------------------------ 0.5/5.5 MB 1.7 MB/s eta 0:00:04
   --- ------------------------------------ 0.5/5.5 MB 1.9 MB/s eta 0:00:03
   ---- ----------------------------------- 0.6/5.5 MB 1.7 MB/s eta 0:00:03
   ---- ----------------------------------- 0.6/5.5 MB 1.7 MB/s eta 0:00:03
   ---- ------------------------------


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [128]:
try:
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callbacks_config()
    prepare_callbacks = PrepareCallbacks(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()

    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train(
        callback_list=callback_list
    )
except Exception as e:
    raise e

[2026-06-06 18:26:13,746: INFO: common: yaml file: config\config.yaml loaded successfully:]
[2026-06-06 18:26:13,748: INFO: common: yaml file: params.yaml loaded successfully:]
[2026-06-06 18:26:13,749: INFO: common: created directory at: artifacts:]
[2026-06-06 18:26:13,750: INFO: common: created directory at: artifacts\prepare_callbacks\checkpoint_dir:]
[2026-06-06 18:26:13,751: INFO: common: created directory at: artifacts\prepare_callbacks\tensorboard_log_dir:]
[2026-06-06 18:26:13,752: INFO: common: created directory at: artifacts\training:]
Found 78 images belonging to 2 classes.
Found 312 images belonging to 2 classes.
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4398 - loss: 15.9181[2026-06-06 18:26:50,555: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_